# Executor Agent
This LangGraph agent is the doer. It receives instructions to target a specific module (or package) and executes automated AST transformations using OpenRewrite. It also has the capability to manually patch files if requested by a downstream Validator Agent.

In [1]:
import os
import subprocess
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

load_dotenv()

if "OPENAI_API_KEY" not in os.environ:
    from getpass import getpass
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

# Using the powerful gpt-4o for complex patching tasks
llm = ChatOpenAI(model="gpt-4o", temperature=0)


### Agent Tools

In [2]:
import subprocess
import os
from langchain_core.tools import tool

TARGET_DIR = "spring-petclinic-1.5.x"

@tool
def run_openrewrite(module_name: str, recipe: str) -> str:
    """Runs OpenRewrite AST transformation. (Monolithic execution, module_name tracked for scope)."""
    print(f"\n>>> [EXECUTING LIVE] OpenRewrite: {recipe}")
    cmd = ["mvn", "org.openrewrite.maven:rewrite-maven-plugin:run", f"-Drewrite.activeRecipes={recipe}"]
    try:
        res = subprocess.run(cmd, cwd=TARGET_DIR, capture_output=True, text=True)
        if res.returncode == 0:
            return f"Success\nOutput:\n{res.stdout[-1500:]}" # return tail to avoid massive context
        return f"Failed!\nError:\n{res.stderr[-2000:]}\nOutput:\n{res.stdout[-2000:]}"
    except Exception as e:
        return str(e)

@tool
def edit_source_code(file_path: str, search_string: str, replace_string: str) -> str:
    """Manual patch tool for the Executor."""
    try:
        if not os.path.exists(file_path):
            return f"Error: File {file_path} does not exist."
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
        if search_string not in content:
            return f"Error: The string was not found EXACTLY."
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(content.replace(search_string, replace_string))
        return f"Successfully patched {file_path}."
    except Exception as e:
        return str(e)

@tool
def run_local_compile(module_name: str) -> str:
    """Runs local compilation: mvn clean compile"""
    print("\n>>> [EXECUTING LIVE] mvn clean compile")
    try:
        res = subprocess.run(["mvn", "clean", "compile"], cwd=TARGET_DIR, capture_output=True, text=True)
        if res.returncode == 0:
            return f"BUILD SUCCESS\n{res.stdout[-1000:]}"
        return f"BUILD FAILED\n{res.stdout[-2000:]}"
    except Exception as e:
        return str(e)

@tool
def run_test_compile(module_name: str) -> str:
    """Runs test compilation: mvn test-compile"""
    print("\n>>> [EXECUTING LIVE] mvn test-compile")
    try:
        res = subprocess.run(["mvn", "test-compile"], cwd=TARGET_DIR, capture_output=True, text=True)
        if res.returncode == 0:
            return f"TEST-COMPILE SUCCESS\n{res.stdout[-1000:]}"
        return f"TEST-COMPILE FAILED\n{res.stdout[-2000:]}"
    except Exception as e:
        return str(e)

@tool
def run_unit_tests(module_name: str) -> str:
    """Runs unit tests: mvn test"""
    print("\n>>> [EXECUTING LIVE] mvn test")
    try:
        res = subprocess.run(["mvn", "test"], cwd=TARGET_DIR, capture_output=True, text=True)
        if res.returncode == 0:
            return f"TEST SUCCESS\n{res.stdout[-1000:]}"
        return f"TEST FAILED\n{res.stdout[-2000:]}"
    except Exception as e:
        return str(e)

@tool
def simulate_git_commit(module_name: str) -> str:
    """Commits the current localized changes securely to the local repo."""
    print(f"\n>>> [EXECUTING LIVE] git commit for {module_name}")
    try:
        if not os.path.exists(os.path.join(TARGET_DIR, ".git")):
            subprocess.run(["git", "init"], cwd=TARGET_DIR, capture_output=True)
        subprocess.run(["git", "add", "."], cwd=TARGET_DIR, capture_output=True)
        res = subprocess.run(["git", "commit", "-m", f"Migrated module {module_name}"], cwd=TARGET_DIR, capture_output=True, text=True)
        return f"Commit output:\n{res.stdout}"
    except Exception as e:
        return str(e)

@tool
def create_migration_pr(module_name: str) -> str:
    """Drafts a PR for the specific module."""
    return f"[MOCK] Created internal PR for 'Migration of {module_name}'"

tools = [run_openrewrite, edit_source_code, run_local_compile, run_test_compile, run_unit_tests, simulate_git_commit, create_migration_pr]


### Agent Logic Definition

In [3]:
system_prompt = """You are the Executor Agent.

Your objective is to carry out AST transformations on specific Java modules to comply with a framework migration plan (e.g., Spring Boot 3 & JDK 17).

Instructions:
1. When given a directive to migrate a module, you MUST ALWAYS trigger `run_openrewrite` using the appropriate module name and the requested recipe (e.g., `org.openrewrite.java.spring.boot3.UpgradeSpringBoot_3_0`).
2. Be concise with your responses. Simply acknowledge the simulation execution, and state that you are ready to pass the state context back to the Validator Agent (which will compile the module and return errors if the Rewrite failed).
3. If, in the future, the Validator Agent feeds you a compiler error, you must determine exactly what syntax or import is broken and construct a manual patch using the `edit_source_code` tool.
"""

agent = create_react_agent(llm, tools, prompt=system_prompt)


### Trial Run

In [4]:
query = "The Planner has commanded: Begin migration of the leaf node. Module target: `petclinic-domain`. Recipe: `org.openrewrite.java.spring.boot3.UpgradeSpringBoot_3_0`"

messages = agent.invoke({"messages": [("user", query)]})

print("Executor Node Output (Awaiting Validator in Master Loop):\n")
print(messages["messages"][-1].content)


[SIMULATED MULTI-MODULE EXECUTION]
Targeting Module: petclinic-domain
Conceptual Command: mvn rewrite:run -Drewrite.activeRecipes=org.openrewrite.java.spring.boot3.UpgradeSpringBoot_3_0 -pl :petclinic-domain
Falling back to global execution due to legacy monolithic layout...
----------------------------------------------------------------------

Executor Node Output (Awaiting Validator in Master Loop):

The simulation of running OpenRewrite on the `petclinic-domain` module has been acknowledged. I am ready to pass the state context back to the Validator Agent.
